# 02 — Silver: очистка через Delta Lake
Превращаем сырьё из bronze в чистый слой silver:- типизация, фильтрация мусора, дедупликация- запись в формате **Delta Lake** (ACID, MERGE, time travel)Запускать после `01_ingestion` (bronze должен быть наполнен).

## 1. Spark-сессия
Delta-расширения подключены в `spark-defaults.conf`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("silver").getOrCreate()
print("Spark:", spark.version)

Spark: 3.5.3


## 2. Клиенты: bronze → silver
Типизация даты регистрации, дедуп по client_id, нормализация города.

In [2]:
clients = spark.read.parquet("s3a://bronze/clients")

clients_clean = (clients
    .filter(F.col("client_id").isNotNull())
    .withColumn("reg_date", F.to_date("reg_date"))
    .withColumn("city", F.trim(F.col("city")))
    .dropDuplicates(["client_id"]))

(clients_clean.write.format("delta").mode("overwrite")
    .save("s3a://silver/clients"))

print("clients  silver ->", clients_clean.count(), "строк")

clients  silver -> 9828 строк


## 3. Счета: bronze → silver
Типизация дат и баланса, дедуп, фильтр битых записей.

In [3]:
accounts = spark.read.parquet("s3a://bronze/accounts")

accounts_clean = (accounts
    .filter(F.col("account_id").isNotNull() & F.col("client_id").isNotNull())
    .withColumn("balance", F.col("balance").cast("double"))
    .withColumn("open_date", F.to_date("open_date"))
    .filter(F.col("balance") >= 0)
    .dropDuplicates(["account_id"]))

(accounts_clean.write.format("delta").mode("overwrite")
    .save("s3a://silver/accounts"))

print("accounts silver ->", accounts_clean.count(), "строк")

accounts silver -> 14171 строк


## 4. Транзакции: bronze → silver
Правила качества:- есть ключ (`tx_id`) и счёт (`account_id`)- сумма положительная- время → настоящий `timestamp`- дедуп по `tx_id` (защита от повторной загрузки)

In [4]:
tx = spark.read.parquet("s3a://bronze/transactions")

tx_clean = (tx
    .filter(F.col("tx_id").isNotNull() & F.col("account_id").isNotNull())
    .filter(F.col("amount") > 0)
    .withColumn("ts", F.to_timestamp("ts"))
    .dropDuplicates(["tx_id"]))

(tx_clean.write.format("delta").mode("overwrite")
    .save("s3a://silver/transactions"))

print("tx       silver ->", tx_clean.count(), "строк")

tx       silver -> 189879 строк


## 5. Проверка silver
Читаем Delta-таблицы обратно, смотрим типы и данные.

In [5]:
for name in ["clients", "accounts", "transactions"]:
    df = spark.read.format("delta").load(f"s3a://silver/{name}")
    print(f"{name:13s} -> {df.count():>7} строк")

print()
print("=== схема транзакций (обратите внимание на ts: timestamp) ===")
spark.read.format("delta").load("s3a://silver/transactions").printSchema()
spark.read.format("delta").load("s3a://silver/transactions").show(5, truncate=False)

clients       ->    9828 строк
accounts      ->   14171 строк
transactions  ->  189879 строк

=== схема транзакций (обратите внимание на ts: timestamp) ===
root
 |-- tx_id: long (nullable = true)
 |-- account_id: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- tx_type: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- status: string (nullable = true)
 |-- ts: timestamp (nullable = true)

+-----+----------+--------+--------+-----------+---------+--------------------------+
|tx_id|account_id|amount  |tx_type |merchant   |status   |ts                        |
+-----+----------+--------+--------+-----------+---------+--------------------------+
|2    |5254      |11882.95|transfer|DNS        |failed   |2026-05-24 00:29:43.415408|
|9    |3478      |21804.82|transfer|Wildberries|pending  |2026-05-23 22:10:39.415668|
|11   |7680      |25051.21|purchase|Лукойл     |failed   |2026-05-24 18:44:39.415773|
|12   |12493     |43422.7 |purchase|Лукойл     |complet

## 6. Time travel — суперсила Delta
Delta хранит историю версий. Можно прочитать таблицу «как было» в прошлой версии.

In [6]:
# история версий таблицы
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, "s3a://silver/transactions")
dt.history().select("version", "timestamp", "operation").show(truncate=False)

# прочитать версию 0 (первая запись)
v0 = spark.read.format("delta").option("versionAsOf", 0).load("s3a://silver/transactions")
print("версия 0:", v0.count(), "строк")

+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|0      |2026-05-24 19:02:03|WRITE    |
+-------+-------------------+---------+

версия 0: 189879 строк


## Готово
Silver наполнен чистыми Delta-таблицами. Дальше — `03_silver_scd2`:история изменений клиентов через `MERGE INTO`.